# DES cluster halo catalog → tSZ map

This notebook paints only the halo one-halo Compton-y signal for the strict `M_interp > 1e13 Msun/h` catalog. The optimized implementation lives in `tsz_pasting.py`; all settings live in `params_tsz.yaml`.

Important physics boundary: the source file does not record an SO mass definition. The output is therefore **conditional on `M_interp` being usable as M200c**, and every saved file records that assumption as provisional. The product has no diffuse/two-halo gas, beam, smoothing, or noise. The projected grid is extended below every catalog pixel-center radius; lower or upper radial extrapolation is forbidden.

In [ ]:
# Run this cell first. Restart the kernel if JAX was previously imported with x64 off.
import os
import sys
os.environ['JAX_ENABLE_X64'] = 'True'
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

import jax
jax.config.update('jax_enable_x64', True)
assert jax.config.jax_enable_x64, 'Restart the kernel: JAX x64 is not enabled.'
print('JAX devices:', jax.devices())

In [ ]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'tsz_pasting.py').exists():
    NOTEBOOK_DIR = Path('/mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/DES_cluster')
sys.path.insert(0, str(NOTEBOOK_DIR))

from tsz_pasting import (load_params, load_tsz_map, preflight_catalog, run_tsz_paste,
                         stratified_row_indices, validate_pair_kernel_against_reference)

PARAMS = NOTEBOOK_DIR / 'params_tsz.yaml'
RUN_FULL_CATALOG = False   # Set True to paint all 3,001,721 selected halos.
SMOKE_MAX_HALOS = 64       # Fast end-to-end check before the production run.
QUICK_NSIDE = 1024         # Production default in YAML is NSIDE=2048.
OVERWRITE = False          # Deliberate safety switch for an existing output path.

overrides = {} if RUN_FULL_CATALOG else {
    'map': {'nside': QUICK_NSIDE},
    'runtime': {'pair_batch_size': 65536, 'halo_chunk_size': SMOKE_MAX_HALOS},
    'output': {'run_name': 'c000_ph000_Mgt1e13_smoke'},
}
max_halos = None if RUN_FULL_CATALOG else SMOKE_MAX_HALOS
cfg = load_params(PARAMS, overrides)
print('Mode:', 'FULL' if RUN_FULL_CATALOG else f'SMOKE ({max_halos} halos)')
print('NSIDE:', cfg['map']['nside'])

## Preflight

This streams the catalog without loading it into memory. It checks the strict cut, row count, finite values, the c000 observer/cosmology radial relation, and interpolation-grid coverage before any expensive profile setup.

In [ ]:
report = preflight_catalog(cfg)
for key in ('source_rows', 'selected_rows', 'mass_min_hmsun', 'mass_max_hmsun',
            'z_min', 'z_max', 'max_distance_redshift_relative_error'):
    print(f'{key}: {report[key]}')

## Paint and save the map

The helper builds the GODMAX pressure/projected-y table once, reuses one fixed-shape JIT kernel, streams halo chunks, and keeps only one dense map. The HDF5 is written atomically and will not be overwritten unless `overwrite=True` is passed explicitly.

In [ ]:
result = run_tsz_paste(
    PARAMS,
    overrides=overrides,
    max_halos=max_halos,
    overwrite=OVERWRITE,
)
print('Saved:', result['path'])
print('Diagnostics:', result['diagnostics'])

In [ ]:
# Read the saved map independently.
import numpy as np

ymap, metadata = load_tsz_map(result['path'])
print('shape / dtype:', ymap.shape, ymap.dtype)
print('min / max / sum:', ymap.min(), ymap.max(), ymap.sum(dtype=np.float64))
print('halos painted:', metadata['n_halos_painted'])
print('mass assumption:', metadata['mass_assumption'])
assert np.all(np.isfinite(ymap))
assert np.all(ymap >= 0.0)

## Optional implementation cross-check

This reuses one profile setup and compares the optimized fixed-batch evaluator with GODMAX's established `get_sim_map` evaluator on the identical pixel package. It also changes the fixed pair-batch size and requires bitwise-identical pair values.

In [ ]:
RUN_REFERENCE_CHECK = False
if RUN_REFERENCE_CHECK:
    reference = validate_pair_kernel_against_reference(
        PARAMS,
        overrides={'map': {'nside': 1024},
                   'runtime': {'halo_chunk_size': 8, 'pixel_batch_size': 8,
                               'pair_batch_size': 64, 'pixel_workers': 1, 'verbose': False}},
        max_halos=8,
        alternate_pair_batch_size=97,
    )
    print(reference)
    assert reference['passed']

In [ ]:
# Optional sky view.
import healpy as hp
hp.mollview(ymap, title='GODMAX halo-only Compton-y', unit='dimensionless', norm='hist')

## Exact null checks

These cheap tests intentionally bypass profile construction. Both a zero-halo run and a zero pressure-amplitude run must write exact all-zero maps.

In [ ]:
null_dir = Path(cfg['output']['directory']) / 'validation_nulls'
null_common = {
    'map': {'nside': 32},
    'runtime': {'halo_chunk_size': 64, 'pixel_batch_size': 64, 'pair_batch_size': 64, 'verbose': False},
    'output': {'directory': str(null_dir), 'compression': None},
}
zero_halo = run_tsz_paste(PARAMS, overrides=null_common, max_halos=0, overwrite=True)
zero_amp_overrides = {**null_common, 'map': {'nside': 32, 'pressure_amplitude': 0.0}}
zero_amp = run_tsz_paste(PARAMS, overrides=zero_amp_overrides, max_halos=64, overwrite=True)
for label, product in [('zero halo', zero_halo), ('zero amplitude', zero_amp)]:
    null_map, _ = load_tsz_map(product['path'])
    assert np.count_nonzero(null_map) == 0
    print(label, 'PASS')

## Grid convergence (required before scientific use)

The full catalog extends to z=2.626, so the YAML grid was widened to z=2.70. Set the flag below to compare the default grid against the denser grid recorded in the parameter file on the same 64 halos. This can be expensive because each grid builds a fresh profile table. NSIDE 2048 is the science default; NSIDE 1024 is a quick-look mode and should not be used for small-scale interpretation until an isolated-halo aperture-Y resolution comparison passes.

In [ ]:
RUN_GRID_CONVERGENCE = False
if RUN_GRID_CONVERGENCE:
    dense = cfg['validation']['dense_grid_overrides']
    validation_base = {
        'map': {'nside': 1024},
        'runtime': {'halo_chunk_size': 64, 'pair_batch_size': 65536, 'verbose': False},
        'output': {'directory': str(Path(cfg['output']['directory']) / 'validation_grid')},
    }
    sample_rows = stratified_row_indices(cfg, 64)
    default_run = run_tsz_paste(PARAMS, overrides=validation_base, row_indices=sample_rows, overwrite=True)
    dense_run = run_tsz_paste(
        PARAMS,
        overrides={**validation_base, 'profiles': {'overrides': dense}},
        row_indices=sample_rows,
        overwrite=True,
    )
    y_default, _ = load_tsz_map(default_run['path'])
    y_dense, _ = load_tsz_map(dense_run['path'])
    active = (y_default != 0) | (y_dense != 0)
    print('sum ratio dense/default:', y_dense.sum(dtype=np.float64) / y_default.sum(dtype=np.float64))
    print('max active-pixel relative difference:',
          np.max(np.abs(y_dense[active] - y_default[active]) / np.maximum(y_dense[active], y_default[active])))